In [1]:
%run ../setup_data.ipynb

No active 'spark' session found to stop.
Starting new Spark session: 'working_with_different_data_type'...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/suanki/Desktop/pyspark-env/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/11 17:18:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/11 17:18:59 WARN Utils: Service 'SparkUI' could not bind on port 4142. Attempting port 4143.


flight path verified (6 items): /home/suanki/Desktop/projects/suanki-pyspark-and-optimization/data/flight-data/csv
retail path verified (7 items): /home/suanki/Desktop/projects/suanki-pyspark-and-optimization/data/retail-data
Loading flight data...
Loading retail data...
+---------+-------------+-----------+
|namespace|    tableName|isTemporary|
+---------+-------------+-----------+
|sujeet_db|   department|      false|
|sujeet_db|       flight|      false|
|sujeet_db|      payment|      false|
|sujeet_db|       retail|      false|
|sujeet_db|sales_metrics|      false|
|sujeet_db| skew_metrics|      false|
|sujeet_db|weather_sales|      false|
+---------+-------------+-----------+



In [2]:
spark = spark

In [3]:
spark.sql("select * from payment").show()

+--------------+------------+--------+
|payment_amount|payment_date|store_id|
+--------------+------------+--------+
|       1200.99|  2018-01-18|       1|
|        189.23|  2018-02-15|       1|
|       3332.23|  2021-02-23|       3|
|       9499.49|  2021-03-10|       3|
|        382.92|  2019-02-18|       1|
|        322.34|  2019-03-29|       2|
|       2929.14|  2020-01-03|       2|
|        499.02|  2020-02-19|       3|
|       3002.43|  2018-02-25|       2|
|        100.99|  2019-03-07|       1|
|        211.65|  2020-02-02|       1|
|        500.73|  2021-01-06|       3|
|        994.11|  2020-03-14|       1|
|        394.93|  2021-01-22|       2|
|         33.43|  2018-03-03|       3|
|       7382.10|  2019-01-11|       2|
+--------------+------------+--------+



# Understanding `GROUPING SETS`, `rollup()`, and `cube()` in PySpark

### 1. What is a Grouping Set?
A standard `groupBy("store_id", "year")` only aggregates at one fixed level (each unique `store_id` + `year` pair). **Grouping Sets** allow you to aggregate at multiple different levels simultaneously—effectively performing the `UNION` of several different `GROUP BY` statements in a single query.

> **⚠️ Critical Warning on `NULL` Values:**  
> Grouping sets, rollups, and cubes use `NULL` in the output columns to represent **grand totals and subtotals** (meaning *"aggregated across all values of this column"*). You must filter out or handle real `NULL` values in your grouping columns beforehand, or you will get inaccurate/ambiguous results.

---

### 2. SQL `GROUPING SETS` vs. DataFrame `.rollup()` & `.cube()`
The `GROUPING SETS` keyword is only available in Spark SQL. In the DataFrame API, you use **`rollup`** and **`cube`**[cite: 4, 5]:

| Operator | API Availability | What Grouping Levels It Computes (Example: Columns `(store_id, year)`) |
| :--- | :--- | :--- |
| **`GROUPING SETS(...)`** | **Spark SQL** | Computes **only the exact combinations** you list inside the parentheses:<br>• `((store_id, year))` $\rightarrow$ Same as normal `GROUP BY store_id, year`<br>• `((store_id), ())` $\rightarrow$ Subtotal per `store_id` + Grand Total `()` |
| **`rollup("store_id", "year")`** | **DataFrame & SQL** | Summarizes **hierarchically** from left to right (order of columns matters!):<br>1. `(store_id, year)` — Subtotal per store per year<br>2. `(store_id, NULL)` — Subtotal per store across all years<br>3. `(NULL, NULL)` — Grand total across all stores and all years |
| **`cube("store_id", "year")`** | **DataFrame & SQL** | Summarizes across **all possible combinations** of columns:<br>1. `(store_id, year)` — Subtotal per store per year<br>2. `(store_id, NULL)` — Subtotal per store across all years<br>3. `(NULL, year)` — Subtotal per year across all stores *(Added by Cube!)*<br>4. `(NULL, NULL)` — Grand total across all stores and all years |

---

### 3. Understanding `grouping_id()`
Spark provides the `grouping_id()` function to tell you which aggregation level a row belongs to. It uses a binary bitmask over your grouping columns `(col1, col2)`:
* Bit = `0` if the column is **included** in the grouping.
* Bit = `1` if the column is **aggregated over** (shown as `NULL`).
* For `(store_id, year)`:
  * `0` (`00` in binary): Grouped by both `(store_id, year)`
  * `1` (`01` in binary): Grouped by `store_id` only (`year` is `NULL`)
  * `2` (`10` in binary): Grouped by `year` only (`store_id` is `NULL`)
  * `3` (`11` in binary): Grand Total (both `store_id` and `year` are `NULL`)

In [4]:
payment_df = spark.sql(""" 
 

  with temp as (
          
          select payment_amount as amount, year(to_date(payment_date, 'yyyy-MM-dd')) as year, store_id as store from payment
          )
  select * from temp

""")


payment_df.show()

+-------+----+-----+
| amount|year|store|
+-------+----+-----+
|1200.99|2018|    1|
| 189.23|2018|    1|
|3332.23|2021|    3|
|9499.49|2021|    3|
| 382.92|2019|    1|
| 322.34|2019|    2|
|2929.14|2020|    2|
| 499.02|2020|    3|
|3002.43|2018|    2|
| 100.99|2019|    1|
| 211.65|2020|    1|
| 500.73|2021|    3|
| 994.11|2020|    1|
| 394.93|2021|    2|
|  33.43|2018|    3|
|7382.10|2019|    2|
+-------+----+-----+



### This is how you resolve without grouping sets

In [5]:
### find sum of amount for store and year wise
from pyspark.sql.functions import *

(

    payment_df
    .groupBy("store","year")
    .agg(sum("amount"))
    .orderBy("store","year")
    .show()
)


+-----+----+-----------+
|store|year|sum(amount)|
+-----+----+-----------+
|    1|2018|    1390.22|
|    1|2019|     483.91|
|    1|2020|    1205.76|
|    2|2018|    3002.43|
|    2|2019|    7704.44|
|    2|2020|    2929.14|
|    2|2021|     394.93|
|    3|2018|      33.43|
|    3|2020|     499.02|
|    3|2021|   13332.45|
+-----+----+-----------+



In [6]:
### find sum of amount for store and year wise 
### sum of amount for each store and year seperately

from pyspark.sql.functions import *

df1 = (

    payment_df
    .groupBy("store","year")
    .agg(sum("amount").alias("total_amount"))
    .orderBy("store","year")
)

df2 = (

    payment_df
    .groupBy("store")
    .agg(sum("amount").alias("total_amount"))
    .withColumn("year", lit(None))
    .orderBy("store")
)

df3 = (

    payment_df
    .groupBy("year")
    .agg(sum("amount").alias("total_amount"))
    .withColumn("store", lit(None))
    .orderBy("year")
)


df1.show()
df2.show()
df3.show()


+-----+----+------------+
|store|year|total_amount|
+-----+----+------------+
|    1|2018|     1390.22|
|    1|2019|      483.91|
|    1|2020|     1205.76|
|    2|2018|     3002.43|
|    2|2019|     7704.44|
|    2|2020|     2929.14|
|    2|2021|      394.93|
|    3|2018|       33.43|
|    3|2020|      499.02|
|    3|2021|    13332.45|
+-----+----+------------+

+-----+------------+----+
|store|total_amount|year|
+-----+------------+----+
|    1|     3079.89|NULL|
|    2|    14030.94|NULL|
|    3|    13864.90|NULL|
+-----+------------+----+

+----+------------+-----+
|year|total_amount|store|
+----+------------+-----+
|2018|     4426.08| NULL|
|2019|     8188.35| NULL|
|2020|     4633.92| NULL|
|2021|    13727.38| NULL|
+----+------------+-----+



In [7]:
df1.union(df2).union(df3).show()

+-----+--------+------------+
|store|    year|total_amount|
+-----+--------+------------+
|    1| 2018.00|     1390.22|
|    1| 2019.00|      483.91|
|    1| 2020.00|     1205.76|
|    2| 2018.00|     3002.43|
|    2| 2019.00|     7704.44|
|    2| 2020.00|     2929.14|
|    2| 2021.00|      394.93|
|    3| 2018.00|       33.43|
|    3| 2020.00|      499.02|
|    3| 2021.00|    13332.45|
|    1| 3079.89|        NULL|
|    2|14030.94|        NULL|
|    3|13864.90|        NULL|
| 2018| 4426.08|        NULL|
| 2019| 8188.35|        NULL|
| 2020| 4633.92|        NULL|
| 2021|13727.38|        NULL|
+-----+--------+------------+



### Lab 1: SQL `GROUPING SETS` (Custom Aggregation Levels)
Suppose we want **three specific totals** in a single table[cite: 4]:
1. `(store_id, year)` — Total sales per store per year
2. `(store_id)` — Subtotal sales for each store across all years
3. `()` — The overall Grand Total across all stores and years[cite: 4]

In [8]:
payment_df.createTempView("payments")

In [9]:
spark.sql("""
 
   
   select store,
          year,
          round(sum(amount), 2) as total_amount,
          grouping_id() as gid
    from payments
    group by store, year
    grouping sets((store,year), (store), ())
    order by store asc nulls last, year asc nulls last

""").show()

+-----+----+------------+---+
|store|year|total_amount|gid|
+-----+----+------------+---+
|    1|2018|     1390.22|  0|
|    1|2019|      483.91|  0|
|    1|2020|     1205.76|  0|
|    1|NULL|     3079.89|  1|
|    2|2018|     3002.43|  0|
|    2|2019|     7704.44|  0|
|    2|2020|     2929.14|  0|
|    2|2021|      394.93|  0|
|    2|NULL|    14030.94|  1|
|    3|2018|       33.43|  0|
|    3|2020|      499.02|  0|
|    3|2021|    13332.45|  0|
|    3|NULL|    13864.90|  1|
| NULL|NULL|    30975.73|  3|
+-----+----+------------+---+



#### Hand-Verification of Output (Lab 1 — SQL `GROUPING SETS` & `rollup`)
Notice that `GROUPING SETS ((store_id, year), (store_id), ())` produces the **exact same hierarchy** as `payments_df.rollup("store_id", "year")`[cite: 4]:

```text
+--------+----+---+------------+
|store_id|year|gid|total_amount|
+--------+----+---+------------+
|1       |2018|0  |1390.22     | <-- Store 1 in 2018: 1200.99 + 189.23 = 1390.22
|1       |2019|0  |483.91      | <-- Store 1 in 2019: 382.92 + 100.99  = 483.91
|1       |2020|0  |1205.76     | <-- Store 1 in 2020: 994.11 + 211.65  = 1205.76
|1       |NULL|1  |3079.89     | <-- Store 1 SUBTOTAL (All Years): 1390.22 + 483.91 + 1205.76 = 3079.89
|2       |2018|0  |3002.43     | <-- Store 2 in 2018: 3002.43
|2       |2019|0  |7704.44     | <-- Store 2 in 2019: 322.34 + 7382.10 = 7704.44
|2       |2020|0  |2929.14     | <-- Store 2 in 2020: 2929.14
|2       |2021|0  |394.93      | <-- Store 2 in 2021: 394.93
|2       |NULL|1  |14030.94    | <-- Store 2 SUBTOTAL (All Years): 3002.43 + 7704.44 + 2929.14 + 394.93
|3       |2018|0  |33.43       | <-- Store 3 in 2018: 33.43
|3       |2020|0  |499.02      | <-- Store 3 in 2020: 499.02
|3       |2021|0  |13332.45    | <-- Store 3 in 2021: 3332.23 + 9499.49 + 500.73 = 13332.45
|3       |NULL|1  |13864.9     | <-- Store 3 SUBTOTAL (All Years): 33.43 + 499.02 + 13332.45 = 13864.90
|NULL    |NULL|3  |30975.73    | <-- GRAND TOTAL (All Stores & Years): 3079.89 + 14030.94 + 13864.90
+--------+----+---+------------+

### Lab 2: DataFrame `.rollup()` vs. `.cube()`
  `.rollup("store_id", "year")` and `.cube("store_id", "year")` in the DataFrame API:
* **`rollup("store_id", "year")`**: Gives Store+Year (`gid=0`), Store Subtotals (`gid=1`), and Grand Total (`gid=3`), **but does NOT give Year Subtotals across all stores**.
* **`cube("store_id", "year")`**: Adds the missing combination `(NULL, year)` (`gid=2`), which gives the **Total per Year across all stores**!

In [10]:
# 1. DataFrame Rollup (Hierarchical: Store -> Year)
print("--- ROLLUP(store, year) ---")
(
    payment_df
    .rollup("store", "year")
    .agg(
        grouping_id().alias("gid"),
        round(sum("amount"), 2).alias("total_amount")
    )
    .orderBy(col("gid").asc(), col("store").asc(), col("year").asc())
    .show(20, False)
)

# 2. DataFrame Cube (All Combinations: Store+Year, Store, Year, Grand Total)
print("--- CUBE(store_id, year) ---")
(
    payment_df
    .cube("store", "year")
    .agg(
        grouping_id().alias("gid"),
        round(sum("amount"), 2).alias("total_amount")
    )
    .orderBy(col("gid").asc(), col("store").asc(), col("year").asc())
    .show(25, False)
)

--- ROLLUP(store, year) ---
+-----+----+---+------------+
|store|year|gid|total_amount|
+-----+----+---+------------+
|1    |2018|0  |1390.22     |
|1    |2019|0  |483.91      |
|1    |2020|0  |1205.76     |
|2    |2018|0  |3002.43     |
|2    |2019|0  |7704.44     |
|2    |2020|0  |2929.14     |
|2    |2021|0  |394.93      |
|3    |2018|0  |33.43       |
|3    |2020|0  |499.02      |
|3    |2021|0  |13332.45    |
|1    |NULL|1  |3079.89     |
|2    |NULL|1  |14030.94    |
|3    |NULL|1  |13864.90    |
|NULL |NULL|3  |30975.73    |
+-----+----+---+------------+

--- CUBE(store_id, year) ---
+-----+----+---+------------+
|store|year|gid|total_amount|
+-----+----+---+------------+
|1    |2018|0  |1390.22     |
|1    |2019|0  |483.91      |
|1    |2020|0  |1205.76     |
|2    |2018|0  |3002.43     |
|2    |2019|0  |7704.44     |
|2    |2020|0  |2929.14     |
|2    |2021|0  |394.93      |
|3    |2018|0  |33.43       |
|3    |2020|0  |499.02      |
|3    |2021|0  |13332.45    |
|1    |NULL|

#### What `.cube("store_id", "year")` Adds (`gid = 2` Rows):
When you inspect the rows where `gid = 2` (`store_id` is `NULL`, `year` is populated) in the `.cube()` output, you see the **yearly totals across all stores combined**[cite: 4]:

| `store_id` | `year` | `gid` | `total_amount` | How Spark Calculated It (`Store 1 + Store 2 + Store 3`) |
| :--- | :--- | :--- | :--- | :--- |
| `NULL` | **`2018`** | `2` | **`4426.08`** | Store 1 (`1390.22`) + Store 2 (`3002.43`) + Store 3 (`33.43`) = **`4426.08`** |
| `NULL` | **`2019`** | `2` | **`8188.35`** | Store 1 (`483.91`) + Store 2 (`7704.44`) = **`8188.35`** |
| `NULL` | **`2020`** | `2` | **`4633.92`** | Store 1 (`1205.76`) + Store 2 (`2929.14`) + Store 3 (`499.02`) = **`4633.92`** |
| `NULL` | **`2021`** | `2` | **`13727.38`** | Store 2 (`394.93`) + Store 3 (`13332.45`) = **`13727.38`** |
| `NULL` | **`NULL`** | `3` | **`30975.73`** | Grand Total: `4426.08 + 8188.35 + 4633.92 + 13727.38` = **`30975.73`** |

# Understanding `pivot()` in PySpark

### 1. What Does `pivot()` Do?
A **Pivot** rotates (transposes) distinct row values from one column into multiple separate columns in the resulting DataFrame. It is always chained between `.groupBy()` and an aggregation function (like `.sum()`, `.avg()`, or `.agg()`):

```python
df.groupBy("row_grouping_col").pivot("column_to_become_headers").agg(...)

# Understanding `pivot()` on `department_df`

### 1. What Does `pivot()` Do on `department_df`?
In `department_df`, we have three columns: `name`, `department`, and `salary`.
* A normal **`groupBy("name", "department").agg(sum("salary"))`** produces a **long (vertical)** table where each `(name, department)` pair is a separate row.
* A **`pivot("department")`** rotates the unique values of the `department` column (`Finance`, `Marketing`, `Sales`) into **separate columns**, creating a wide cross-tabulation matrix.

```python
department_df.groupBy("name").pivot("department").agg(sum("salary"))

In [11]:
department_df = spark.sql("select * from sujeet_db.department")

department_df.show()

+---------+----------+------+
|     name|department|salary|
+---------+----------+------+
|     Jeff| Marketing|  3000|
|    Kumar| Marketing|  2000|
|  Michael|     Sales|  4600|
|   Robert|     Sales|  4100|
|    Maria|   Finance|  3000|
|    James|     Sales|  3000|
|    James|     Sales|  3000|
|    Scott|   Finance|  3300|
|      Jen|   Finance|  3900|
|     Saif|     Sales|  4100|
|   Daniel|      NULL|  NULL|
|Christian|      NULL|  NULL|
+---------+----------+------+



In [12]:
department_df.groupBy("name").pivot("department").agg(sum("salary")).show()

+---------+----+-------+---------+-----+
|     name|null|Finance|Marketing|Sales|
+---------+----+-------+---------+-----+
|    Scott|NULL|   3300|     NULL| NULL|
|    James|NULL|   NULL|     NULL| 6000|
|Christian|NULL|   NULL|     NULL| NULL|
|      Jen|NULL|   3900|     NULL| NULL|
|  Michael|NULL|   NULL|     NULL| 4600|
|    Kumar|NULL|   NULL|     2000| NULL|
|   Daniel|NULL|   NULL|     NULL| NULL|
|     Saif|NULL|   NULL|     NULL| 4100|
|    Maria|NULL|   3000|     NULL| NULL|
|   Robert|NULL|   NULL|     NULL| 4100|
|     Jeff|NULL|   NULL|     3000| NULL|
+---------+----+-------+---------+-----+



In [13]:
(
    department_df
    .groupBy("name")
    .pivot("department", ["Finance", "Marketing", "Sales"])
    .agg(sum("salary"))
    .orderBy("name")
    .na.fill(0)
).show()

+---------+-------+---------+-----+
|     name|Finance|Marketing|Sales|
+---------+-------+---------+-----+
|Christian|      0|        0|    0|
|   Daniel|      0|        0|    0|
|    James|      0|        0| 6000|
|     Jeff|      0|     3000|    0|
|      Jen|   3900|        0|    0|
|    Kumar|      0|     2000|    0|
|    Maria|   3000|        0|    0|
|  Michael|      0|        0| 4600|
|   Robert|      0|        0| 4100|
|     Saif|      0|        0| 4100|
|    Scott|   3300|        0|    0|
+---------+-------+---------+-----+



In [14]:
# 3. Pivot with Multiple Aggregations (See how James has count=2 and sum=6000 in Sales!)
pivoted_multi_df = (
    department_df
    .where(col("department").isNotNull())
    .groupBy("name")
    .pivot("department", ["Finance", "Marketing", "Sales"])
    .agg(
        sum("salary").alias("sum"),
        count("salary").alias("cnt"),
        collect_list("salary").alias("list")
    )
    .na.fill(0)
    .orderBy("name")
)

pivoted_multi_df.show()

+-------+-----------+-----------+------------+-------------+-------------+--------------+---------+---------+------------+
|   name|Finance_sum|Finance_cnt|Finance_list|Marketing_sum|Marketing_cnt|Marketing_list|Sales_sum|Sales_cnt|  Sales_list|
+-------+-----------+-----------+------------+-------------+-------------+--------------+---------+---------+------------+
|  James|          0|          0|          []|            0|            0|            []|     6000|        2|[3000, 3000]|
|   Jeff|          0|          0|          []|         3000|            1|        [3000]|        0|        0|          []|
|    Jen|       3900|          1|      [3900]|            0|            0|            []|        0|        0|          []|
|  Kumar|          0|          0|          []|         2000|            1|        [2000]|        0|        0|          []|
|  Maria|       3000|          1|      [3000]|            0|            0|            []|        0|        0|          []|
|Michael|       

In [15]:
# 4. What if you just want 1 summary row with each Department as a column?
# Group by an empty list groupBy() and pivot on "department":
dept_summary_row = (
    department_df
    .groupBy()
    .pivot("department", ["Finance", "Marketing", "Sales"])
    .agg(sum("salary"))
)

print("--- Single-Row Department Totals Matrix ---")
dept_summary_row.show()

--- Single-Row Department Totals Matrix ---
+-------+---------+-----+
|Finance|Marketing|Sales|
+-------+---------+-----+
|  10200|     5000|18800|
+-------+---------+-----+



# Understanding `unpivot()` in PySpark (on `department_df`)

### 1. What Does `unpivot()` Do?
`unpivot()` is the inverse operation of `pivot()`:
* **`pivot()` (Long $\rightarrow$ Wide)**: Turns row values (`Finance`, `Marketing`, `Sales`) into separate columns.
* **`unpivot()` (Wide $\rightarrow$ Long)**: Takes separate columns (`Finance`, `Marketing`, `Sales`) and stacks them back into rows under two new columns:
  1. **`variableColumnName`** (e.g., `"department"`): Holds the name of the column the value came from.
  2. **`valueColumnName`** (e.g., `"salary"`): Holds the numeric value that was inside that column cell.

---

### 2. Three Ways to Unpivot in PySpark

| Method | Spark Version | Syntax Example |
| :--- | :--- | :--- |
| **`df.unpivot()`** | **Spark 3.4+ / 4.x** *(Recommended)* | `pivoted_df.unpivot(ids=["name"], values=["Finance", "Marketing", "Sales"], variableColumnName="department", valueColumnName="salary")` |
| **`df.melt()`** | **Spark 3.4+ / 4.x** *(Alias for `unpivot`)* | `pivoted_df.melt(ids=["name"], values=["Finance", "Marketing", "Sales"], variableColumnName="department", valueColumnName="salary")` |
| **SQL `stack()`** | **All Spark Versions** | `pivoted_df.select("name", expr("stack(3, 'Finance', Finance, 'Marketing', Marketing, 'Sales', Sales) as (department, salary)"))` |

> **Important Note on Aggregation Loss:**
> When we pivoted `department_df`, the two `James` rows in `Sales` (`3000` and `3000`) were aggregated into a single cell (`6000`). When you `unpivot()` that wide table back to long format, `James` will have **one** row with `6000` (because `unpivot` cannot un-sum an already aggregated total).

In [16]:
department_df.show()

+---------+----------+------+
|     name|department|salary|
+---------+----------+------+
|     Jeff| Marketing|  3000|
|    Kumar| Marketing|  2000|
|  Michael|     Sales|  4600|
|   Robert|     Sales|  4100|
|    Maria|   Finance|  3000|
|    James|     Sales|  3000|
|    James|     Sales|  3000|
|    Scott|   Finance|  3300|
|      Jen|   Finance|  3900|
|     Saif|     Sales|  4100|
|   Daniel|      NULL|  NULL|
|Christian|      NULL|  NULL|
+---------+----------+------+



In [17]:
pivoted_dept_df = (
    department_df
    .where(col("department").isNotNull())
    .groupBy("name")
    .pivot("department", ["Finance", "Marketing", "Sales"])
    .agg(sum("salary"))
    .orderBy("name")
)

print("--- Wide Table (pivoted_dept_df) ---")
pivoted_dept_df.show()



--- Wide Table (pivoted_dept_df) ---
+-------+-------+---------+-----+
|   name|Finance|Marketing|Sales|
+-------+-------+---------+-----+
|  James|   NULL|     NULL| 6000|
|   Jeff|   NULL|     3000| NULL|
|    Jen|   3900|     NULL| NULL|
|  Kumar|   NULL|     2000| NULL|
|  Maria|   3000|     NULL| NULL|
|Michael|   NULL|     NULL| 4600|
| Robert|   NULL|     NULL| 4100|
|   Saif|   NULL|     NULL| 4100|
|  Scott|   3300|     NULL| NULL|
+-------+-------+---------+-----+



In [ ]:

unpivoted_all_df = pivoted_dept_df.unpivot(
    ids=["name"],                              # Column(s) to keep fixed as row identifiers
    values=["Finance", "Marketing", "Sales"],  # Columns to rotate back into rows
    variableColumnName="department",           # Name for the new column holding old headers
    valueColumnName="salary"                   # Name for the new column holding cell values
)


unpivoted_all_df.show()

unpivoted_all_df.dropna().show()

+-------+----------+------+
|   name|department|salary|
+-------+----------+------+
|  James|   Finance|  NULL|
|  James| Marketing|  NULL|
|  James|     Sales|  6000|
|   Jeff|   Finance|  NULL|
|   Jeff| Marketing|  3000|
|   Jeff|     Sales|  NULL|
|    Jen|   Finance|  3900|
|    Jen| Marketing|  NULL|
|    Jen|     Sales|  NULL|
|  Kumar|   Finance|  NULL|
|  Kumar| Marketing|  2000|
|  Kumar|     Sales|  NULL|
|  Maria|   Finance|  3000|
|  Maria| Marketing|  NULL|
|  Maria|     Sales|  NULL|
|Michael|   Finance|  NULL|
|Michael| Marketing|  NULL|
|Michael|     Sales|  4600|
| Robert|   Finance|  NULL|
| Robert| Marketing|  NULL|
+-------+----------+------+
only showing top 20 rows
+-------+----------+------+
|   name|department|salary|
+-------+----------+------+
|  James|     Sales|  6000|
|   Jeff| Marketing|  3000|
|    Jen|   Finance|  3900|
|  Kumar| Marketing|  2000|
|  Maria|   Finance|  3000|
|Michael|     Sales|  4600|
| Robert|     Sales|  4100|
|   Saif|     Sales|  4

In [19]:
spark.stop()